## <font color='blue'>Generative AI and LLMs for Natural Language Processing</font>
## <font color='blue'>Efficient Fine-Tuning of LLMs with LoRA for Text Sentiment Analysis</font>

## Installing and Loading Packages

In [1]:
!pip install -q -U watermark

In [2]:
!pip install -q accelerate==1.9.0 peft==0.16.0 bitsandbytes==0.46.1 transformers==4.54.0

In [3]:
!pip install -q trl==0.20.0 gradio==5.38.2 protobuf scipy==1.16.0

In [4]:
!pip install -q sentencepiece==0.2.0 tokenizers==0.21.2 datasets==4.0.0

In [6]:
# Imports
import os
import torch
import datasets
import pandas as pd
from datasets import load_dataset
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          HfArgumentParser,
                          TrainingArguments,
                          pipeline,
                          logging)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set the log level to CRITICAL.
logging.set_verbosity(logging.CRITICAL)

In [ ]:
# Check the GPU model.
if torch.cuda.is_available():
    print('Número de GPUs:', torch.cuda.device_count())
    print('Modelo GPU:', torch.cuda.get_device_name(0))
    print('Total Memória [GB] da GPU:',torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
#GPU memory reset
from numba import cuda
device = cuda.get_current_device()
device.reset()

In [ ]:
# Define the name of the dataset.
nome_dataset = "dataset.csv"

In [ ]:
# Load the dataset using the Hugging Face Datasets library.
dataset_carregado = load_dataset('csv', data_files = nome_dataset, delimiter = ',')

In [ ]:
# Data Loaded in Dictionary Format
dataset_carregado

https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

In [ ]:
# Name of the pre-trained LLM repository
repositorio_hf = "NousResearch/Llama-2-7b-chat-hf"

In [ ]:
# Name of the new model
modelo_dsa = "novo_modelo_dsa"

## Defining the Configuration Parameters

In [ ]:
# LoRA Parameters
lora_r = 32
lora_alpha = 16
lora_dropout = 0.1

The parameters above are from LoRA (Low-Rank Adaptation), the base part of QLoRA (Quantized Low-Rank Adaptation), a technique used to adapt language models efficiently. Above the LoRA parameters, we place the quantization parameters, thus defining QLoRA.

Let's describe each of the parameters:

**lora_r**: This parameter represents the "rank" in Low-Rank Adaptation (LoRA). A value of 32 means that the original model's weight matrix will be approximated by two smaller matrices whose product has a maximum rank of 32. Essentially, this reduces computational complexity and the number of parameters to be trained during adaptation, while maintaining model effectiveness.

**lora_alpha**: This is a scaling factor applied to LoRA weight updates during training. A value of 16 indicates that weight updates will be scaled by this factor. This parameter is important because it allows fine control over the magnitude of weight updates, which can affect the speed and effectiveness of model adaptation.

**lora_dropout**: This parameter represents the "dropout" rate applied during model adaptation. The value 0.1 means that 10% of the units will be randomly discarded (or "turned off") during training. Dropout is a common technique to avoid overfitting in neural networks, ensuring that the model does not become overly dependent on any specific part of the training data.

In [ ]:
# bitsandbytes parameters (QLoRa)
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

The parameters above are for the bitsandbytes library, a tool for optimizing the training of machine learning models, particularly to reduce memory usage and accelerate training. Here is the explanation of each parameter:

**use_4bit**: This parameter indicates whether 4-bit quantization should be used or not. By setting True, this means the model will use a 4-bit representation for weights during training. This significantly reduces the amount of memory required, allowing training of larger models or reducing hardware requirements.

**bnb_4bit_compute_dtype**: This is the data type used for calculations during training when 4-bit quantization is active. The value float16 means calculations will be done using 16-bit floating-point numbers. This is generally used to balance computational efficiency and numerical precision.

**bnb_4bit_quant_type**: Specifies the type of quantization to be used. The value nf4 is a specific quantization type developed by bitsandbytes, optimized for efficiency and effectiveness in training deep learning models. This quantization type is designed to maintain model precision while reducing memory requirements.

**use_nested_quant**: Indicates whether a nested quantization technique will be used. False means this technique will not be employed. Nested quantization can be used to further reduce memory usage by applying different levels of quantization to different parts of the model, but it can be more complex to implement and manage.

These parameters are used to configure how the deep learning model will handle the representation and calculation of weights during training, aiming to optimize memory usage and accelerate the training process.

In [ ]:
# Fine-tuning parameters
output_dir = "saida"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1
warmup_ratio = 0.03

The parameters above are used to configure the fine-tuning process of machine learning models, especially natural language models. Let's describe each one:

**output_dir**: Specifies the directory where training results will be saved.

**num_train_epochs**: The number of training epochs. The value 1 means the model will pass once through the entire training dataset.

**fp16**: Indicates whether to use mixed-precision training with 16-bit floating-point (FP16). The value True means yes, which can accelerate training and reduce memory usage while maintaining acceptable precision.

**bf16**: Similar to fp16, but for the bfloat16 format. False means it will not be used. The bfloat16 format is another way to reduce memory usage and accelerate training, with slightly different impacts on precision. We can use fp16 or bf16, but we cannot use both simultaneously.

**per_device_train_batch_size**: Training batch size per device. The value 4 indicates that each training device (such as a GPU) will process 4 examples per batch.

**per_device_eval_batch_size**: Evaluation batch size per device, also set to 4.

**gradient_accumulation_steps**: Number of steps for gradient accumulation before performing a parameter update. The value 1 means there is no accumulation (each step results in an update).

**gradient_checkpointing**: Enables gradient checkpointing, a technique to reduce memory usage at the cost of slightly longer training time. True indicates it is enabled.

**max_grad_norm**: Maximum norm for gradient clipping. The value 0.3 is a value that helps avoid the gradient explosion problem in trainings.

**learning_rate**: Initial learning rate. The value 2e-4 is a common value for fine-tuning, providing a balance between learning speed and stability.

**weight_decay**: Weight decay rate, used for regularization. The value 0.001 is a value that helps prevent overfitting.

**optim**: The optimizer used. "paged_adamw_32bit" is a variant of AdamW optimized for memory efficiency.

**lr_scheduler_type**: Type of learning rate scheduler. The value "cosine" indicates the use of the cosine scheduler, which adjusts the learning rate following a cosine curve.

**max_steps**: Maximum number of training steps. The value -1 means training will continue until the number of epochs is reached.

**warmup_ratio**: Proportion of the total training steps used for linear warmup of the learning rate. The value 0.03 means that 3% of the initial training will be used to gradually increase the learning rate.

These parameters are essential to efficiently configure the fine-tuning process, directly impacting the quality of the trained model, training time, and computational resource usage.

In [ ]:
# Grouping sequences into batches of the same length
group_by_length = True
save_steps = 0
logging_steps = 400

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
The parameters above are used to configure certain aspects of the training process of machine learning models, especially language models. They are related to how sequences are grouped into batches and how training progress is logged and saved. Let's detail each one:

**group_by_length**: This parameter indicates whether sequences should be grouped by length when forming training batches. When True, this means training will group sequences of similar lengths together in a batch. This is an efficient practice because it reduces the amount of padding needed. Padding is used to ensure all sequences in a batch have the same length, but it can be a waste of computational resources. Grouping sequences of similar lengths minimizes this waste.

**save_steps**: Specifies the frequency with which the trained model should be saved. A value of 0 indicates that the model will not be saved automatically based on a number of steps. Instead, the model can be saved at the end of each training epoch or manually.

**logging_steps**: Defines the frequency with which log information should be recorded. The value 400 means that the training process will log information such as training loss, evaluation metrics, among others, every 400 training steps. This is useful for monitoring training progress and for fine-tuning hyperparameters.

In [ ]:
# Data precision for training
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

The code above is related to configuring the data type (dtype) for computation during training of neural network models using the PyTorch library. Let's analyze each part of the code:

**torch**: It is a reference to the PyTorch library, a library for machine learning and neural networks.

**bnb_4bit_compute_dtype**: This is a variable that stores a string representing the desired data type for computation. The parameter bnb_4bit_compute_dtype was set to "float16", indicating that computation should be performed using 16-bit floating-point numbers.

**getattr**: It is a built-in Python function used to get an attribute of an object. In this case, it is being used to get an attribute from the PyTorch library based on the value of the string stored in bnb_4bit_compute_dtype.

What happens here is that getattr(torch, bnb_4bit_compute_dtype) retrieves the 16-bit floating-point data type (torch.float16) from the PyTorch library, based on the value of bnb_4bit_compute_dtype. Then, this data type is assigned to the variable compute_dtype.

The use of compute_dtype in training neural network models has important implications:

**Memory Efficiency**: Using float16 instead of more common data types like float32 can significantly reduce memory usage, allowing training of larger models or running more processes in parallel.

**Computation Speed**: Many modern GPUs have optimizations for float16 calculations, which can accelerate training.

**Precision**: Although float16 may be less precise than float32, it is often sufficiently precise for neural network model training tasks.

In [ ]:
# Defining quantization parameters
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [ ]:
# Loading the pre-trained base model
modelo = AutoModelForCausalLM.from_pretrained(repositorio_hf,
                                              quantization_config = bnb_config,
                                              device_map = "auto")

In [ ]:
# We will not use cache
modelo.config.use_cache = False
modelo.config.pretraining_tp = 1

In [ ]:
# Loading the tokenizer of the base model
tokenizador = AutoTokenizer.from_pretrained(repositorio_hf, trust_remote_code = True)
tokenizador.pad_token = tokenizador.eos_token
tokenizador.padding_side = "right"

In [ ]:
# Loading LoRA configuration
peft_config = LoraConfig(lora_alpha = lora_alpha,
                         lora_dropout = lora_dropout,
                         r = lora_r,
                         bias = "none",
                         task_type = "CAUSAL_LM")

https://github.com/huggingface/peft/blob/main/src/peft/tuners/lora/config.py

In [ ]:
# Defining training parameters
training_arguments = TrainingArguments(output_dir = output_dir,
                                       num_train_epochs = num_train_epochs,
                                       per_device_train_batch_size = per_device_train_batch_size,
                                       gradient_accumulation_steps = gradient_accumulation_steps,
                                       optim = optim,
                                       save_steps = save_steps,
                                       logging_steps = logging_steps,
                                       learning_rate = learning_rate,
                                       weight_decay = weight_decay,
                                       fp16 = fp16,
                                       bf16 = bf16,
                                       max_grad_norm = max_grad_norm,
                                       max_steps = max_steps,
                                       warmup_ratio = warmup_ratio,
                                       group_by_length = group_by_length,
                                       lr_scheduler_type = lr_scheduler_type)

In [ ]:
# Defining Supervised Fine-Tuning Parameters
sft_config = SFTConfig(per_device_train_batch_size = 2,
                       gradient_accumulation_steps = 4,
                       warmup_steps = 5,
                       num_train_epochs = 1,
                       learning_rate = 2e-4,
                       fp16 = True,
                       logging_steps = 1,
                       optim = "adamw_8bit",
                       weight_decay = 0.01,
                       lr_scheduler_type = "linear",
                       seed = 3407,
                       output_dir = "outputs",
                       report_to = "none",
                       dataset_text_field = "train",
                       packing = False)

In [ ]:
# Defining Supervised Fine-Tuning Parameters (requires the general parameters in sft_config above)
dsa_trainer = SFTTrainer(model = modelo,
                         train_dataset = dataset_carregado['train'],
                         peft_config = peft_config,
                         args = sft_config)

https://huggingface.co/docs/trl/en/sft_trainer

> Model Training with Fine Tuning

In [ ]:
%%time
dsa_trainer.train()

In [ ]:
# Saving the trained model
dsa_trainer.model.save_pretrained(modelo_dsa)

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [ ]:
# New input text
prompt = "It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement"

https://huggingface.co/docs/transformers/en/main_classes/pipelines

In [ ]:
# Sentiment Analysis Pipeline with the Fine-Tuned Model
pipe = pipeline(task = "text-generation",
                model = modelo,
                tokenizer = tokenizador,
                max_length = 200)

In [ ]:
# Runs the pipeline and extracts the result
resultado = pipe(f"<s>[INST] {prompt} [/INST]")

In [ ]:
print(resultado)

In [ ]:
print(resultado[0]['generated_text'])

In [ ]:
# Frees memory
del modelo
del pipe
del dsa_trainer
import gc
gc.collect()

In [ ]:
# Loads the model in fp16 and merges with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(repositorio_hf,
                                                  low_cpu_mem_usage = True,
                                                  return_dict = True,
                                                  torch_dtype = torch.float16,
                                                  device_map = "auto")

In [ ]:
# Creates the final model
modelo_dsa_final = PeftModel.from_pretrained(base_model, modelo_dsa)

In [ ]:
# Merges and unloads the model
modelo_dsa_final = modelo_dsa_final.merge_and_unload()

In [ ]:
# Loads the tokenizer
tokenizador_dsa = AutoTokenizer.from_pretrained(repositorio_hf, trust_remote_code = True)
tokenizador_dsa.pad_token = tokenizador_dsa.eos_token
tokenizador_dsa.padding_side = "right"

In [ ]:
# Saves model and tokenizer
modelo_dsa_final.save_pretrained('novo_modelo-dsa-llm-projeto2')
tokenizador_dsa.save_pretrained('novo_modelo-dsa-llm-projeto2')

In [ ]:
# New input text
prompt = "It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement"

In [ ]:
# Creates the pipeline
pipe = pipeline(task = "text-generation",
                model = modelo_dsa_final,
                tokenizer = tokenizador_dsa,
                max_length = 200)

In [ ]:
# Runs the pipeline and extracts the result
resultado = pipe(f"<s>[INST] {prompt} [/INST]")

In [ ]:
print(resultado)

In [ ]:
# Let's not just classify the sentiment.
# Let's generate positive and/or negative text from the initial evaluation (text).
print(resultado[0]['generated_text'])

In [ ]:
# Frees GPU memory
from numba import cuda
device = cuda.get_current_device()
device.reset()

# End